# Facial Expression Recognition — EfficientNet-B3 160px Stratified Notebook

Included changes:
- SFEW removed.
- Checkpoints renamed to prevent accidental resume from old runs.
- Raw samples store dataset identity + modality.
- Train/val split is stratified by dataset + class.
- Validation uses clean transforms only.
- Surprise augmentation is train-only and RGB-source only.
- Mixed precision training enabled for faster T4 training.
- DataLoader workers/persistent workers enabled.
- Last gradient accumulation step is flushed.
- Scheduler state and best epoch are restored correctly.


In [ ]:
import random, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter, deque

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import (
    DataLoader, Dataset, ConcatDataset, WeightedRandomSampler
)
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

warnings.filterwarnings('ignore')

def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Faster on fixed image size. Use deterministic=True only if exact reproducibility matters more than speed.
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

set_seed(42, deterministic=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")


In [ ]:
# ── Dataset paths ──────────────────────────────────────────────
RAF_TRAIN_CSV   = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/train_labels.csv'
RAF_TEST_CSV    = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/test_labels.csv'
RAF_TRAIN_DIR   = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train'
RAF_TEST_DIR    = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test'

FERPLUS_TRAIN   = '/kaggle/input/datasets/arnabkumarroy02/ferplus/train'
FERPLUS_VAL     = '/kaggle/input/datasets/arnabkumarroy02/ferplus/validation'
FERPLUS_TEST    = '/kaggle/input/datasets/arnabkumarroy02/ferplus/test'

AFFECTNET_TRAIN = '/kaggle/input/datasets/mstjebashazida/affectnet/archive (3)/Train'
AFFECTNET_TEST  = '/kaggle/input/datasets/mstjebashazida/affectnet/archive (3)/Test'

# SFEW excluded — movie frames hurt webcam-focused generalization.

# ── Classes ────────────────────────────────────────────────────
CLASSES      = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

RAFDB_MAP = {
    1: 'surprise', 2: 'fear',    3: 'disgust',
    4: 'happy',    5: 'sad',     6: 'angry',
    7: 'neutral'
}

AFFECTNET_FOLDER_MAP = {
    'anger':    'angry',   'angry':     'angry',
    'disgust':  'disgust', 'fear':      'fear',
    'happy':    'happy',   'happiness': 'happy',
    'neutral':  'neutral', 'sad':       'sad',
    'sadness':  'sad',     'surprise':  'surprise',
}
FOLDER_CLASS_MAP = {
    'angry': 'angry',
    'anger': 'angry',

    'disgust': 'disgust',

    'fear': 'fear',

    'happy': 'happy',
    'happiness': 'happy',

    'neutral': 'neutral',

    'sad': 'sad',
    'sadness': 'sad',

    'surprise': 'surprise',
    'surprised': 'surprise',
    'suprise': 'surprise',   # FERPlus typo folder
}

# ── Training config ────────────────────────────────────────────
IMG_SIZE     = 160
BATCH_SIZE   = 32
EPOCHS       = 120
PATIENCE     = 15
GRAD_ACCUM   = 2

CAP_DOMINANT = 10000
CAP_SURPRISE = 8000
AUG_TARGET   = 8000
VAL_SPLIT    = 0.15

# DataLoader speed settings for Kaggle T4.
NUM_WORKERS = 4

# ── Checkpoints — renamed to avoid old-run resume collision ────
RUN_NAME     = 'b3_160_stratified_cleanval_v3'
CKPT_BEST    = f'/kaggle/working/best_model_{RUN_NAME}.pth'
CKPT_LATEST  = f'/kaggle/working/latest_checkpoint_{RUN_NAME}.pth'
DEPLOY_MODEL = f'/kaggle/working/fer_deploy_{RUN_NAME}.pth'

# ── Normalization ──────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Classes  : {CLASSES}")
print(f"IMG_SIZE : {IMG_SIZE} | BATCH: {BATCH_SIZE} | GRAD_ACCUM: {GRAD_ACCUM}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"EPOCHS: {EPOCHS} | PATIENCE: {PATIENCE} | VAL_SPLIT: {VAL_SPLIT}")
print(f"Run name: {RUN_NAME}")


In [ ]:
# ── RGB train ─────────────────────────────────────────────────
rgb_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.15,
        hue=0.02
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.07, 0.07),
        scale=(0.92, 1.08)
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.12, scale=(0.01, 0.06))
])

# ── Grayscale train — FERPlus ─────────────────────────────────
gray_train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.07, 0.07),
        scale=(0.92, 1.08)
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.08, scale=(0.01, 0.05))
])

# ── Surprise augmentation — RGB-source surprise only ───────────
surprise_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=18),
    transforms.ColorJitter(
        brightness=0.35,
        contrast=0.35,
        saturation=0.20,
        hue=0.04
    ),
    transforms.RandomAffine(
        degrees=8,
        translate=(0.10, 0.10),
        scale=(0.88, 1.12)
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08))
])

# ── Val/test — clean only ─────────────────────────────────────
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

gray_val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print(f"Transforms ready at IMG_SIZE={IMG_SIZE}")
print("Train RGB   : moderate augmentation")
print("Train gray  : no color jitter")
print("Val/test    : clean resize only")


In [ ]:
class TaggedDataset(Dataset):
    """
    Sample format:
      (dataset_name, modality, path, label)

    dataset_name:
      'rafdb', 'affectnet', 'ferplus'

    modality:
      'rgb' or 'gray'

    The dataset name is used for stratified splitting and diagnostics.
    The modality decides which transform to apply.
    """
    def __init__(self, tagged_samples, rgb_tf, gray_tf):
        self.samples = tagged_samples
        self.rgb_tf = rgb_tf
        self.gray_tf = gray_tf

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        dataset_name, modality, path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        tf = self.gray_tf if modality == 'gray' else self.rgb_tf
        if tf:
            img = tf(img)
        return img, label


class SurpriseAugDataset(Dataset):
    """
    Synthetic surprise samples.
    Training only. Never validation.
    Samples are (path, label), taken only from RGB-source surprise images.
    """
    def __init__(self, samples, transform, n):
        self.samples = samples
        self.transform = transform
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        path, label = self.samples[idx % len(self.samples)]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


print("Dataset classes defined.")


In [ ]:
"""
Raw sample format:
  (dataset_name, modality, path, label)

Examples:
  ('rafdb',     'rgb',  path, label)
  ('affectnet', 'rgb',  path, label)
  ('ferplus',   'gray', path, label)
"""

print("Collecting raw samples with dataset + modality tags...")
print("=" * 65)

# ── RAF-DB raw samples ─────────────────────────────────────────
raf_df = pd.read_csv(RAF_TRAIN_CSV)

label_map = {
    rafdb_num: CLASS_TO_IDX[emotion]
    for rafdb_num, emotion in RAFDB_MAP.items()
}

raf_raw = []
class_seen = Counter()

for _, row in raf_df.iterrows():
    label_idx = label_map[int(row['label'])]
    path = os.path.join(
        RAF_TRAIN_DIR,
        str(int(row['label'])),
        row['image']
    )

    if not os.path.exists(path):
        continue

    cap = CAP_SURPRISE if IDX_TO_CLASS[label_idx] == 'surprise' else CAP_DOMINANT

    if class_seen[label_idx] < cap:
        raf_raw.append(('rafdb', 'rgb', path, label_idx))
        class_seen[label_idx] += 1

print(f"RAF-DB raw     : {len(raf_raw):,}")

# ── AffectNet raw samples ──────────────────────────────────────
affectnet_raw = []

for folder in os.listdir(AFFECTNET_TRAIN):
    folder_lower = folder.lower()

    if folder_lower not in AFFECTNET_FOLDER_MAP:
        continue

    emotion = AFFECTNET_FOLDER_MAP[folder_lower]
    label_idx = CLASS_TO_IDX[emotion]
    folder_path = os.path.join(AFFECTNET_TRAIN, folder)

    if not os.path.isdir(folder_path):
        continue

    files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    random.shuffle(files)

    for f in files[:CAP_DOMINANT]:
        affectnet_raw.append(
            ('affectnet', 'rgb', os.path.join(folder_path, f), label_idx)
        )

print(f"AffectNet raw  : {len(affectnet_raw):,}")

# ── Generic folder collector for FERPlus/test sets ─────────────
def collect_folder_samples(root_dir, dataset_name, modality, cap=None):
    samples = []

    for folder in os.listdir(root_dir):
        folder_lower = folder.lower()

        if folder_lower not in CLASS_TO_IDX:
            continue

        label_idx = CLASS_TO_IDX[folder_lower]
        folder_path = os.path.join(root_dir, folder)

        if not os.path.isdir(folder_path):
            continue

        files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        random.shuffle(files)

        if cap:
            files = files[:cap]

        for f in files:
            samples.append(
                (dataset_name, modality, os.path.join(folder_path, f), label_idx)
            )

    return samples

# ── FERPlus raw samples ────────────────────────────────────────
ferplus_raw = collect_folder_samples(
    FERPLUS_TRAIN,
    dataset_name='ferplus',
    modality='gray',
    cap=CAP_DOMINANT
)

# FERPlus validation is used as part of the training pool.
# We create our own clean validation split later.
ferplus_raw += collect_folder_samples(
    FERPLUS_VAL,
    dataset_name='ferplus',
    modality='gray',
    cap=2000
)

print(f"FERPlus raw    : {len(ferplus_raw):,}")

# ── Combine all raw samples ────────────────────────────────────
all_tagged = raf_raw + affectnet_raw + ferplus_raw
random.shuffle(all_tagged)

print("=" * 65)
print(f"Total raw pool : {len(all_tagged):,}")

# ── Global class distribution ──────────────────────────────────
raw_counts = Counter(label for _, _, _, label in all_tagged)

print("\nRaw class distribution:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:<10}: {raw_counts[i]:>6,}")

# ── Dataset distribution ───────────────────────────────────────
dataset_counts = Counter(dataset for dataset, _, _, _ in all_tagged)

print("\nRaw dataset distribution:")
for dataset, count in sorted(dataset_counts.items()):
    print(f"  {dataset:<10}: {count:>6,}")

# ── Dataset × class distribution ───────────────────────────────
print("\nDataset × class distribution:")
for dataset in sorted(dataset_counts.keys()):
    print(f"\n{dataset.upper()}")
    ds_counts = Counter(
        label for ds, _, _, label in all_tagged
        if ds == dataset
    )
    for i, cls in enumerate(CLASSES):
        print(f"  {cls:<10}: {ds_counts[i]:>6,}")


In [ ]:
"""
Stratified split by dataset + class.

Why:
  A plain random split can accidentally create a validation set with too much
  AffectNet, too little RAF-DB, or weak minority-class representation.

Stratification key:
  (dataset_name, class_label)
"""

print("Creating stratified train/val split by dataset + class...")
print("=" * 65)

rng = np.random.default_rng(42)

# Group indices by (dataset, label)
groups = {}
for idx, (dataset, modality, path, label) in enumerate(all_tagged):
    key = (dataset, label)
    groups.setdefault(key, []).append(idx)

train_indices = []
val_indices = []

for key, idxs in groups.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)

    if len(idxs) >= 2:
        n_val = max(1, int(round(len(idxs) * VAL_SPLIT)))
        n_val = min(n_val, len(idxs) - 1)  # keep at least 1 train sample
    else:
        n_val = 0

    val_indices.extend(idxs[:n_val].tolist())
    train_indices.extend(idxs[n_val:].tolist())

rng.shuffle(train_indices)
rng.shuffle(val_indices)

train_tagged = [all_tagged[i] for i in train_indices]
val_tagged   = [all_tagged[i] for i in val_indices]

print(f"Train raw : {len(train_tagged):,}")
print(f"Val raw   : {len(val_tagged):,}")
print(f"Actual val ratio: {len(val_tagged) / len(all_tagged):.2%}")

def print_split_distribution(name, samples):
    print(f"\n{name} distribution")
    print("-" * 45)

    ds_counts = Counter(ds for ds, _, _, _ in samples)
    cls_counts = Counter(label for _, _, _, label in samples)

    print("By dataset:")
    for ds in sorted(ds_counts.keys()):
        print(f"  {ds:<10}: {ds_counts[ds]:>6,}")

    print("\nBy class:")
    for i, cls in enumerate(CLASSES):
        print(f"  {cls:<10}: {cls_counts[i]:>6,}")

print_split_distribution("TRAIN", train_tagged)
print_split_distribution("VAL", val_tagged)

# ── Build train/val datasets with correct transforms ───────────
train_base_ds = TaggedDataset(
    train_tagged,
    rgb_tf=rgb_train_tf,
    gray_tf=gray_train_tf
)

val_ds = TaggedDataset(
    val_tagged,
    rgb_tf=val_tf,
    gray_tf=gray_val_tf
)

# ── Surprise augmentation — train only, RGB-source only ─────────
surprise_idx = CLASS_TO_IDX['surprise']

current_surprise = sum(
    1 for _, _, _, label in train_tagged
    if label == surprise_idx
)

needed = max(0, AUG_TARGET - current_surprise)

surprise_samples = [
    (path, label)
    for dataset, modality, path, label in train_tagged
    if label == surprise_idx and modality == 'rgb'
]

surprise_aug_ds = None

if needed > 0 and len(surprise_samples) > 0:
    surprise_aug_ds = SurpriseAugDataset(
        surprise_samples,
        surprise_train_tf,
        needed
    )

    print(f"\nSurprise aug: {current_surprise} → {current_surprise + needed} (+{needed} synthetic)")
    print("NOTE: Surprise augmentation added to TRAIN only, not validation.")
    print("NOTE: Surprise augmentation uses RGB-source surprise images only.")
elif needed > 0 and len(surprise_samples) == 0:
    print("\nWARNING: Surprise augmentation needed but no RGB surprise samples found.")
else:
    print(f"\nSurprise aug not needed. Current train surprise count: {current_surprise}")

# ── Final train dataset ────────────────────────────────────────
if surprise_aug_ds is not None:
    train_ds = ConcatDataset([train_base_ds, surprise_aug_ds])
else:
    train_ds = train_base_ds

print(f"\nFinal train ds : {len(train_ds):,}")
print(f"Final val ds   : {len(val_ds):,}")
print("\nVal set is clean: real images only, no augmentation, stratified by dataset + class.")


In [ ]:
# ── Sanity check: validation distribution by dataset × class ───
print("\nValidation dataset × class distribution:")
print("=" * 65)

for dataset_name in sorted(set(ds for ds, _, _, _ in val_tagged)):
    print(f"\n{dataset_name.upper()}")

    ds_counts = Counter(
        label
        for ds, _, _, label in val_tagged
        if ds == dataset_name
    )

    for i, cls in enumerate(CLASSES):
        print(f"  {cls:<10}: {ds_counts[i]:>5,}")

print("\nIf every dataset has all/most classes represented, the split is healthy.")


In [ ]:
# ── Collect train labels for sampler ───────────────────────────
train_labels = []

# Base train samples
for dataset, modality, path, label in train_tagged:
    train_labels.append(label)

# Surprise augmented samples
if surprise_aug_ds is not None:
    train_labels.extend([surprise_idx] * len(surprise_aug_ds))

assert len(train_labels) == len(train_ds), \
    f"MISMATCH: {len(train_labels)} labels vs {len(train_ds)} samples"

print(f"Label list verified: {len(train_labels):,} == {len(train_ds):,} ✓")

# ── Weighted sampler ───────────────────────────────────────────
label_counts = Counter(train_labels)

class_counts_arr = np.array(
    [label_counts[i] for i in range(NUM_CLASSES)],
    dtype=float
)

if np.any(class_counts_arr == 0):
    missing = [
        CLASSES[i]
        for i, count in enumerate(class_counts_arr)
        if count == 0
    ]
    raise ValueError(f"Missing classes in train split: {missing}")

class_weights_arr = 1.0 / class_counts_arr
sample_weights = [class_weights_arr[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nClass distribution in train:")
print("=" * 45)
for i, cls in enumerate(CLASSES):
    print(f"  {cls:<10}: {label_counts[i]:>6,}")
print("=" * 45)

# ── DataLoader kwargs ──────────────────────────────────────────
pin_memory = torch.cuda.is_available()
loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': pin_memory
}

if NUM_WORKERS > 0:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2
    })

# ── Train / Val loaders ────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    **loader_kwargs
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

# ── Test loaders — untouched until final evaluation ────────────

# RAF-DB test
raf_test_samples = []
raf_test_df = pd.read_csv(RAF_TEST_CSV)

for _, row in raf_test_df.iterrows():
    label_idx = label_map[int(row['label'])]
    path = os.path.join(
        RAF_TEST_DIR,
        str(int(row['label'])),
        row['image']
    )

    if os.path.exists(path):
        raf_test_samples.append(('rafdb', 'rgb', path, label_idx))

raf_test_ds = TaggedDataset(
    raf_test_samples,
    rgb_tf=val_tf,
    gray_tf=gray_val_tf
)

# AffectNet test
affectnet_test_samples = []

for folder in os.listdir(AFFECTNET_TEST):
    folder_lower = folder.lower()

    if folder_lower not in AFFECTNET_FOLDER_MAP:
        continue

    emotion = AFFECTNET_FOLDER_MAP[folder_lower]
    label_idx = CLASS_TO_IDX[emotion]
    folder_path = os.path.join(AFFECTNET_TEST, folder)

    if not os.path.isdir(folder_path):
        continue

    for f in os.listdir(folder_path):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            affectnet_test_samples.append(
                ('affectnet', 'rgb', os.path.join(folder_path, f), label_idx)
            )

affectnet_test_ds = TaggedDataset(
    affectnet_test_samples,
    rgb_tf=val_tf,
    gray_tf=gray_val_tf
)

# FERPlus test
ferplus_test_samples = collect_folder_samples(
    FERPLUS_TEST,
    dataset_name='ferplus',
    modality='gray',
    cap=None
)

ferplus_test_ds = TaggedDataset(
    ferplus_test_samples,
    rgb_tf=val_tf,
    gray_tf=gray_val_tf
)

raf_test_loader = DataLoader(
    raf_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

affectnet_test_loader = DataLoader(
    affectnet_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

ferplus_test_loader = DataLoader(
    ferplus_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(
    f"Test — RAF-DB:{len(raf_test_ds):,} "
    f"AffectNet:{len(affectnet_test_ds):,} "
    f"FERPlus:{len(ferplus_test_ds):,}"
)
print(f"DataLoader workers: {NUM_WORKERS}")


In [ ]:
class SimpleHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout=0.35):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


def build_model(num_classes):
    m = models.efficientnet_b3(weights='IMAGENET1K_V1')

    for param in m.parameters():
        param.requires_grad = True

    in_features = m.classifier[1].in_features
    m.classifier = SimpleHead(in_features, num_classes)

    return m


def get_layerwise_optimizer(model, base_lr=2e-4, decay=0.8):
    """
    Layerwise LR decay:
      classifier: highest LR
      deeper feature blocks: medium LR
      earlier feature blocks/stem: lower LR
    """
    param_groups = []

    # Classifier head
    head_params = [
        p for n, p in model.named_parameters()
        if 'classifier' in n and p.requires_grad
    ]

    param_groups.append({
        'params': head_params,
        'lr': base_lr,
        'name': 'classifier'
    })

    # EfficientNet feature blocks
    for depth, block_idx in enumerate(range(8, -1, -1)):
        block_name = f'features.{block_idx}'

        block_params = [
            p for n, p in model.named_parameters()
            if block_name in n
            and 'classifier' not in n
            and p.requires_grad
        ]

        if block_params:
            param_groups.append({
                'params': block_params,
                'lr': base_lr * (decay ** (depth + 1)),
                'name': block_name
            })

    return optim.AdamW(param_groups, weight_decay=1e-4)


model = build_model(NUM_CLASSES).to(device)
optimizer = get_layerwise_optimizer(model, base_lr=2e-4, decay=0.8)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())

print("Model: EfficientNet-B3 + SimpleHead")
print(f"Trainable: {trainable:,} / {total_p:,}")

print("\nLayerwise LR groups:")
for pg in optimizer.param_groups:
    n = sum(p.numel() for p in pg['params'])
    print(f"  {pg['name']:<22} lr={pg['lr']:.2e}  params={n:,}")


In [ ]:
# ── Loss ───────────────────────────────────────────────────────
# No class weights. WeightedRandomSampler already handles imbalance.
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# ── Scheduler ──────────────────────────────────────────────────
# Resume-safe. No fixed total_steps problem like OneCycleLR.
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=2,
    eta_min=1e-7
)

# ── AMP mixed precision ────────────────────────────────────────
USE_AMP = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Loss      : CrossEntropyLoss(label_smoothing=0.05)")
print("Scheduler : CosineAnnealingWarmRestarts(T_0=20, T_mult=2)")
print(f"AMP       : {USE_AMP}")
print(f"Grad accum: {GRAD_ACCUM} (effective batch={BATCH_SIZE * GRAD_ACCUM})")


In [ ]:
def mixup_data(x, y, alpha=0.15):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix_data(x, y, alpha=0.7):
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size(0), device=x.device)

    W, H = x.size(3), x.size(2)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    x_out = x.clone()
    x_out[:, :, y1:y2, x1:x2] = x[rand_index, :, y1:y2, x1:x2]

    lam_adj = 1 - ((x2 - x1) * (y2 - y1)) / (W * H)

    return x_out, y, y[rand_index], lam_adj


def mixed_criterion(crit, pred, y_a, y_b, lam):
    if not isinstance(lam, torch.Tensor):
        lam = torch.tensor(lam, device=pred.device, dtype=pred.dtype)

    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, grad_accum=2):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad(set_to_none=True)
    last_batch_idx = len(loader) - 1

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            r = np.random.rand()

            if r < 0.15:
                # MixUp — 15%
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.15)
                outputs = model(images)
                raw_loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)

            elif r < 0.25:
                # CutMix — 10%
                images, y_a, y_b, lam = cutmix_data(images, labels, alpha=0.7)
                outputs = model(images)
                raw_loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)

            else:
                # Clean — 75%
                outputs = model(images)
                raw_loss = criterion(outputs, labels)

            loss = raw_loss / grad_accum

        scaler.scale(loss).backward()

        is_update_step = ((batch_idx + 1) % grad_accum == 0)
        is_last_batch = (batch_idx == last_batch_idx)

        if is_update_step or is_last_batch:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += raw_loss.item() * images.size(0)

        # Approximate because MixUp/CutMix labels are blended.
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()

    return running_loss / total, 100.0 * correct / total


def eval_epoch(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


print("Train/eval functions ready.")
print("Mix: 15% MixUp + 10% CutMix + 75% clean")
print("AMP enabled:", USE_AMP)
print("Train accuracy is approximate due to MixUp/CutMix. Validation accuracy is the real signal.")


In [ ]:
best_acc = 0.0
best_epoch = 0
no_improve = 0
start_epoch = 1

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# ── Resume from latest checkpoint ──────────────────────────────
if os.path.exists(CKPT_LATEST):
    print("Resuming from latest checkpoint...")
    ckpt = torch.load(CKPT_LATEST, map_location=device)

    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])

    if 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])

    start_epoch = ckpt['epoch'] + 1
    best_acc = ckpt['best_acc']
    best_epoch = ckpt.get('best_epoch', 0)
    no_improve = ckpt.get('no_improve', 0)
    history = ckpt['history']

    print(f"  Resumed epoch     : {start_epoch}")
    print(f"  Best acc so far   : {best_acc:.2f}% (epoch {best_epoch})")
    print(f"  No-improve count  : {no_improve}")
    print(f"  Head LR           : {optimizer.param_groups[0]['lr']:.2e}")
else:
    print("Starting fresh.")

print(f"\nEpochs: {EPOCHS} | Patience: {PATIENCE}\n")
print(f"{'Epoch':<8} {'~Train':>8} {'T-Loss':>8} "
      f"{'Val Acc':>9} {'V-Loss':>8}  {'HeadLR':>9}  Status")
print("-" * 72)

for epoch in range(start_epoch, EPOCHS + 1):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scheduler,
        scaler,
        GRAD_ACCUM
    )

    val_loss, val_acc = eval_epoch(model, val_loader, criterion)

    head_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    is_best = val_acc > best_acc
    status = ""

    if is_best:
        best_acc = val_acc
        best_epoch = epoch
        no_improve = 0
        status = "✓ BEST"

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'best_acc': best_acc,
            'best_epoch': best_epoch,
            'classes': CLASSES,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'architecture': 'efficientnet_b3',
            'run_name': RUN_NAME,
            'history': history,
        }, CKPT_BEST)
    else:
        no_improve += 1

    # Atomic latest checkpoint
    tmp_path = CKPT_LATEST + '.tmp'

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'val_acc': val_acc,
        'best_acc': best_acc,
        'best_epoch': best_epoch,
        'no_improve': no_improve,
        'classes': CLASSES,
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
        'architecture': 'efficientnet_b3',
        'run_name': RUN_NAME,
        'history': history,
    }, tmp_path)

    os.replace(tmp_path, CKPT_LATEST)

    print(f"[{epoch:03d}/{EPOCHS}]  "
          f"{train_acc:>7.2f}%  {train_loss:>7.4f}  "
          f"{val_acc:>8.2f}%  {val_loss:>7.4f}  "
          f"{head_lr:>9.2e}  {status}")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\n{'=' * 72}")
print(f"BEST VAL ACCURACY : {best_acc:.2f}%  (epoch {best_epoch})")
print("Val set is clean and stratified by dataset + class.")


In [ ]:
def evaluate_loader(model, loader, name):
    model.eval()

    preds = []
    trues = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(images)

            _, predicted = outputs.max(1)

            preds.extend(predicted.cpu().numpy())
            trues.extend(labels.numpy())

    acc = 100 * sum(p == l for p, l in zip(preds, trues)) / len(trues)

    print(f"\n{'=' * 55}")
    print(f"{name} — {acc:.2f}%")
    print(f"{'=' * 55}")
    print(classification_report(
        trues,
        preds,
        target_names=CLASSES,
        digits=3,
        zero_division=0
    ))

    return acc, preds, trues


print("Loading best checkpoint...")
ckpt = torch.load(CKPT_BEST, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Best epoch: {ckpt['epoch']} | Val: {ckpt['val_acc']:.2f}%\n")

raf_acc, raf_p, raf_t = evaluate_loader(
    model,
    raf_test_loader,
    "RAF-DB TEST"
)

aff_acc, aff_p, aff_t = evaluate_loader(
    model,
    affectnet_test_loader,
    "AFFECTNET TEST"
)

ferp_acc, ferp_p, ferp_t = evaluate_loader(
    model,
    ferplus_test_loader,
    "FERPLUS TEST"
)

all_p = raf_p + aff_p + ferp_p
all_t = raf_t + aff_t + ferp_t

comb = 100 * sum(p == l for p, l in zip(all_p, all_t)) / len(all_t)

print(f"\n{'=' * 55}")
print("FINAL SUMMARY")
print(f"{'=' * 55}")
print(f"  RAF-DB     : {raf_acc:.2f}%  ← webcam proxy")
print(f"  AffectNet  : {aff_acc:.2f}%")
print(f"  FERPlus    : {ferp_acc:.2f}%")
print(f"  Combined   : {comb:.2f}%")
print(f"  Val clean  : {best_acc:.2f}%")
print(f"{'=' * 55}")


In [ ]:
# ── RAF-DB confusion matrix — most relevant for webcam proxy ───
cm = confusion_matrix(raf_t, raf_p, labels=list(range(NUM_CLASSES)))

row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(
    cm.astype(float),
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cmap='Blues',
    ax=axes[0]
)

axes[0].set_title(f'RAF-DB Confusion Matrix — {raf_acc:.2f}%')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cmap='Blues',
    ax=axes[1]
)

axes[1].set_title('Normalized RAF-DB Confusion Matrix')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_rafdb_stratified.png', dpi=150)
plt.show()

# ── Training curves ────────────────────────────────────────────
ep = range(1, len(history['val_acc']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(ep, history['train_loss'], label='Train', alpha=0.6)
axes[0].plot(ep, history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_acc'], label='Train (~approx)', alpha=0.6)
axes[1].plot(ep, history['val_acc'], label='Val (honest)', linewidth=2)
axes[1].axhline(
    y=best_acc,
    color='red',
    linestyle='--',
    label=f'Best {best_acc:.2f}%'
)
axes[1].set_title('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves_stratified.png', dpi=150)
plt.show()


In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': CLASSES,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'architecture': 'efficientnet_b3',
    'run_name': RUN_NAME,
    'val_accuracy': best_acc,
    'raf_accuracy': raf_acc,
    'affectnet_accuracy': aff_acc,
    'ferplus_accuracy': ferp_acc,
    'combined_accuracy': comb,
    'best_epoch': best_epoch,
}, DEPLOY_MODEL)

print("=" * 60)
print("FILES SAVED")
print("=" * 60)

for path in [
    CKPT_BEST,
    CKPT_LATEST,
    DEPLOY_MODEL,
    '/kaggle/working/confusion_matrix_rafdb_stratified.png',
    '/kaggle/working/training_curves_stratified.png'
]:
    if os.path.exists(path):
        mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {os.path.basename(path):<45} {mb:.1f} MB")

print(f"""
Key numbers:
  Val accuracy      : {best_acc:.2f}%  (clean stratified validation)
  RAF-DB test       : {raf_acc:.2f}%  (webcam proxy)
  AffectNet test    : {aff_acc:.2f}%
  FERPlus test      : {ferp_acc:.2f}%
  Combined test     : {comb:.2f}%
  Best epoch        : {best_epoch}

Download from Output panel:
  - {os.path.basename(DEPLOY_MODEL)}
  - {os.path.basename(CKPT_BEST)}
  - {os.path.basename(CKPT_LATEST)}
  - confusion_matrix_rafdb_stratified.png
  - training_curves_stratified.png
""")


In [ ]:
class EmotionSmoother:
    def __init__(self, classes, window=5):
        self.classes = classes
        self.buffer = deque(maxlen=window)

    def update(self, probs_dict):
        self.buffer.append(probs_dict)
        return {
            cls: round(
                sum(frame[cls] for frame in self.buffer) / len(self.buffer),
                1
            )
            for cls in self.classes
        }

    def reset(self):
        self.buffer.clear()


EMOJI = {
    'angry': '😠',
    'disgust': '🤢',
    'fear': '😨',
    'happy': '😊',
    'neutral': '😐',
    'sad': '😢',
    'surprise': '😲'
}


def load_deploy_model(path):
    ckpt = torch.load(path, map_location=device)
    m = build_model(ckpt['num_classes'])
    m.load_state_dict(ckpt['model_state_dict'])
    m = m.to(device).eval()

    print(
        f"Loaded {ckpt.get('run_name', 'model')} | "
        f"val={ckpt.get('val_accuracy', 0):.2f}% | "
        f"raf={ckpt.get('raf_accuracy', 0):.2f}% | "
        f"epoch={ckpt.get('best_epoch', 'unknown')}"
    )

    return m, ckpt['classes'], ckpt['img_size']


def predict_emotion(image_input, model, class_names, img_size=160, smoother=None):
    import cv2

    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])

    if isinstance(image_input, str):
        img = Image.open(image_input).convert('RGB')
    else:
        img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))

    tensor = tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(tensor)
            probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    raw = {
        class_names[i]: round(float(probs[i]) * 100, 1)
        for i in range(len(class_names))
    }

    final = smoother.update(raw) if smoother else raw
    top = max(final, key=final.get)

    print(f"\n{top.upper()} {EMOJI.get(top, '')} ({final[top]:.1f}%)")

    for emotion, prob in sorted(final.items(), key=lambda x: -x[1]):
        print(f"  {emotion:<10} {'█' * int(prob / 3):<33} {prob:.1f}%")

    return top, final[top], final


# Smoke test on a RAF-DB test image
print("Smoke test on RAF-DB test image...")

for folder in os.listdir(RAF_TEST_DIR):
    folder_path = os.path.join(RAF_TEST_DIR, folder)

    if not os.path.isdir(folder_path):
        continue

    files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if files:
        test_path = os.path.join(folder_path, files[0])
        print(f"Test image: {test_path}")
        predict_emotion(
            test_path,
            model,
            CLASSES,
            IMG_SIZE,
            EmotionSmoother(CLASSES, window=1)
        )
        break
